In [ ]:
# Cell 2 — Install Dependencies

!pip install -q -U \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-text-splitters \
    sentence-transformers \
    transformers \
    accelerate \
    bitsandbytes \
    faiss-cpu \
    pypdf

In [ ]:
# Cell 4 — Google Drive

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Cell 5 — Project Path

PROJECT_PATH = "/content/drive/MyDrive/RAG_Project"

DATA_PATH = f"{PROJECT_PATH}/data"
VECTOR_DB_PATH = f"{PROJECT_PATH}/vectorstore"

print(DATA_PATH)
print(VECTOR_DB_PATH)

/content/drive/MyDrive/RAG_Project/data
/content/drive/MyDrive/RAG_Project/vectorstore


In [ ]:
#  Imports

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda
)
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# Function: Create RAG Project Directories

import os

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(VECTOR_DB_PATH, exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [ ]:
# Cell 5 — Project Path

PDF_PATH = "/content/drive/MyDrive/RAG_Project/Development-And-Testing-Of-Biogas-petrol-Blend-As-An-Alternative-Fuel-For-Spark-Ignition-Engine.pdf"

print(PDF_PATH)

/content/drive/MyDrive/RAG_Project/Development-And-Testing-Of-Biogas-petrol-Blend-As-An-Alternative-Fuel-For-Spark-Ignition-Engine.pdf


In [ ]:
# Cell 8 — Load PDF

loader = PyPDFLoader(PDF_PATH)

docs = loader.load()

print("Number of pages:", len(docs))
print(docs[0].page_content[:500])

Number of pages: 8
INTERNATIONAL JOURNAL OF SCIENTIFIC & TECHNOLOGY RESEARCH VOLUME 4, ISSUE 09, SEPTEMBER 2015    ISSN 2277-8616 
179 
IJSTR©2015 
www.ijstr.org 
Development And Testing Of Biogas-Petrol Blend 
As An Alternative Fuel For Spark Ignition Engine 
 
Awogbemi, Omojola, Adeyemo, Sunday Babatunde 
 
Abstract: This research is on the development and testing of a biogas -petrol blend to run a spark ignition engine. A20:80 ratio 
biogas:petrol blend was developed as an alternative fuel for spark ignition en


In [ ]:
# Cell 9 — Text Splitting

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

splits = splitter.split_documents(docs)

print("Number of chunks:", len(splits))

Number of chunks: 63


In [ ]:

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={
        "device": "cuda"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from google.colab import userdata
import os

os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Smithing Rag"
os.environ["LANGSMITH_ENDPOINT"] = "https://apac.api.smith.langchain.com"

In [ ]:
# Cell 11 — Create Vector Store

vectorstore = FAISS.from_documents(
    splits,
    embeddings
)

print("FAISS vector store created.")

FAISS vector store created.


In [ ]:
# Cell 12 — Retriever

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    }
)

print("Retriever ready.")

Retriever ready.


In [ ]:
# Prompting

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Answer ONLY from the provided context,preferd hinglish language. "
        "If the answer is not found in the context, say you don't know."
    ),
    (
        "human",
        "Question: {question}\n\n"
        "Context:\n{context}"
    )
])

In [ ]:
# ugging Face LLM

from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

llm_pipeline = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    device=0,
    max_new_tokens=512,
    temperature=0.0,
    do_sample=False,
    return_full_text=False
)

llm = HuggingFacePipeline(
    pipeline=llm_pipeline
)

print("LLM loaded.")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

LLM loaded.


In [ ]:
#  Format Retrieved Documents

def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

In [ ]:
#  RAG Retrieval Pipeline

parallel = RunnableParallel({
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough()
})

In [ ]:
# Complete Rag chain
chain = parallel | prompt | llm | StrOutputParser()

In [ ]:
# Cell 18 — Ask Question

question = "COmparisons between Diesel fuel and Oil blends?"

answer = chain.invoke(question)

print(answer)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 

Table 5: Comparison of Torque and Indicated Power 
between Petrol and Biogas-Petrol Blend 
Table 6: Comparison of Brake Power and Brake Thermal Efficiency 
between Petrol and Biogas-Petrol Blend

Assistant: System: The provided context does not directly compare Diesel fuel and Oil blends, especially transterifiedSheanut Oil blends. It focuses on comparisons between biogas-petrol blends and traditional petrol, discussing their performance in engines. Therefore, I cannot provide a comparison between Diesel fuel and Oil blends based on the given information. System: Don't know. The context provided does not contain any direct comparison between Diesel fuel and oil blends, including transterifiedSheanut Oil blends. The information available is related to biogas-petrol blends and their performance in engines. System: Don't know. The context does not provide a comparison between Diesel fuel and oil blends, focusing instead on the performance of biogas-petrol blends in engines.
